In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

In [ ]:
df = pd.read_csv('brestCancer.csv')
df = df.loc[:, ~df.columns.str.contains('unnamed', case=False)]
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)

le = LabelEncoder()
y = le.fit_transform(df['diagnosis'])          # M=1, B=0
X = df.drop(columns=['diagnosis'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Train shape:", X_train_s.shape)
print("Test shape :", X_test_s.shape)
print("Classes    :", dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)

acc_lr  = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr  = recall_score(y_test, y_pred_lr)
f1_lr   = f1_score(y_test, y_pred_lr)

print(f"Accuracy : {acc_lr:.4f}")
print(f"Precision: {prec_lr:.4f}")
print(f"Recall   : {rec_lr:.4f}")
print(f"F1-Score : {f1_lr:.4f}")

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
y_pred_knn = knn.predict(X_test_s)

acc_knn  = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn)
rec_knn  = recall_score(y_test, y_pred_knn)
f1_knn   = f1_score(y_test, y_pred_knn)

print(f"Accuracy : {acc_knn:.4f}")
print(f"Precision: {prec_knn:.4f}")
print(f"Recall   : {rec_knn:.4f}")
print(f"F1-Score : {f1_knn:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, y_pred, title, cmap in zip(
    axes,
    [y_pred_lr, y_pred_knn],
    ['Logistic Regression', 'KNN (k=5)'],
    ['Blues', 'Greens']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, cbar=False,
                xticklabels=['Benign','Malignant'],
                yticklabels=['Benign','Malignant'], ax=ax)
    ax.set_title(f'{title} – Confusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.tight_layout()
plt.show()

In [ ]:
# Reduce to 2 components for visualisation
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_s)
X_test_2d  = pca.transform(X_test_s)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, clf, title, cmap in zip(
    axes,
    [LogisticRegression(random_state=42, max_iter=1000),
     KNeighborsClassifier(n_neighbors=5)],
    ['Logistic Regression', 'KNN (k=5)'],
    ['RdBu', 'PiYG']
):
    clf.fit(X_train_2d, y_train)

    x_min, x_max = X_test_2d[:, 0].min() - 1, X_test_2d[:, 0].max() + 1
    y_min, y_max = X_test_2d[:, 1].min() - 1, X_test_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap)
    colors = ['steelblue', 'tomato']
    for cls, label in enumerate(['Benign', 'Malignant']):
        mask = y_test == cls
        ax.scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
                   c=colors[cls], label=label, edgecolors='k',
                   linewidths=0.4, s=40, alpha=0.8)
    ax.set_title(f'{title} – Decision Boundary (PCA 2D)')
    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
metrics = pd.DataFrame({
    'Model'    : ['Logistic Regression', 'KNN (k=5)'],
    'Accuracy' : [round(acc_lr,  4), round(acc_knn,  4)],
    'Precision': [round(prec_lr, 4), round(prec_knn, 4)],
    'Recall'   : [round(rec_lr,  4), round(rec_knn,  4)],
    'F1-Score' : [round(f1_lr,   4), round(f1_knn,   4)],
})
display(metrics)